# GPU session — run all cells top to bottom

**This machine is deleted permanently when its clock runs out.** Anything not pushed to
GitHub is gone. Every stage here commits and pushes its results immediately, so an
interruption costs one stage rather than the session.

The long work runs under `nohup`, so **it survives your browser disconnecting or the tab
closing**. You can shut the laptop and come back.

Budget: ~5 GPU-hours of work against ~20 available. There is slack for reruns.

## 1. Verify the GPU is real before spending time on it

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
REPO = "https://github.com/mzkaell/vla-where-does-language-die.git"
BRANCH = "phase1-scaffold"

import os, subprocess, pathlib
if not pathlib.Path("vla-where-does-language-die").exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO], check=True)
os.chdir(pathlib.Path.home() / "vla-where-does-language-die"
         if (pathlib.Path.home() / "vla-where-does-language-die").exists()
         else "vla-where-does-language-die")
print("cwd:", os.getcwd())
!git log --oneline -3

## 3. GitHub credentials — so results survive the machine

Uses `getpass`, so the token is **not** written into this notebook or its saved output.
Create a fine-grained PAT with *Contents: read and write* on this repo only.

If you skip this, everything still runs but **nothing is saved off-machine** — you would
have to download `results/` by hand before the clock runs out.

In [ ]:
import getpass, os, subprocess

token = getpass.getpass("GitHub token (blank to skip pushing): ").strip()
if token:
    url = f"https://{token}@github.com/mzkaell/vla-where-does-language-die.git"
    subprocess.run(["git", "remote", "set-url", "origin", url], check=True)
    os.environ["GIT_PUSH"] = "1"
    os.environ["GIT_NAME"] = "mzkaell"
    os.environ["GIT_EMAIL"] = "schmalzmichael50@gmail.com"
    # Verify now rather than discovering it fails after hours of compute.
    ok = subprocess.run(["git", "push", "--dry-run", "origin", "HEAD"],
                        capture_output=True, text=True)
    print("push check:", "OK" if ok.returncode == 0 else "FAILED\n" + ok.stderr)
else:
    os.environ["GIT_PUSH"] = "0"
    print("WARNING: results will NOT be pushed. Download results/ manually before shutdown.")

## 4. Install (~5 min)

In [ ]:
!pip install -q -e ".[vla,dev]"
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU VISIBLE")

## 5. Launch everything in the background

`nohup` detaches the process, so **closing the tab or losing the connection will not kill
it**. The next cell tails the log; re-run that cell any time to check progress.

The script is **resumable** — if anything dies, just run this cell again and it skips the
stages that already produced results.

In [ ]:
import subprocess, os
env = dict(os.environ, DEVICE="cuda", TRIALS="40")
subprocess.Popen(
    "nohup bash scripts/gpu_session.sh >> gpu_session.log 2>&1 &",
    shell=True, env=env,
)
print("launched in background -- safe to close the tab")

## 6. Monitor (re-run this cell whenever)

In [ ]:
!tail -40 gpu_session.log

In [ ]:
# What has been produced and saved so far?
!ls -1 results/ 2>/dev/null
print("--- pushed? ---")
!git log --oneline origin/HEAD~5..origin/HEAD 2>/dev/null || git log --oneline -5

## 7. Before the machine dies — verify results are actually off-machine

Do not trust the log. Open the repo on GitHub and confirm `results/` contains the new
`loc_*` directories. If pushing failed, download them now:

In [ ]:
# Fallback: bundle results small enough to download through the browser.
!tar czf results_backup.tar.gz results/ paper/ && ls -lh results_backup.tar.gz
print("Right-click -> Download this file from the Jupyter file browser.")